# ☕ Does Coffee Add Years to Your Life? — What Public Data Says

**Question:** Do people who drink coffee live longer than people who don't?

**Data:** U.S. National Health and Nutrition Examination Survey (NHANES),
2003–2018 (8 cycles), linked to death records through 31 Dec 2019.
All data is free and public, from the CDC.

## Step 1 — Download the raw data

For each survey cycle we download:

| File | What it contains |
|------|------------------|
| DEMO | Age, sex, race, income, education, survey weights |
| DR1IFF / DR2IFF | Every food & drink eaten on recall day 1 and day 2, with USDA food codes |
| DR1TOT | Total calories eaten on day 1 |
| SMQ | Smoking history |
| BMX | Measured weight, height and BMI |
| MORT (2019) | Whether the person died by 2019, and months of follow-up |

Raw files are saved untouched to Google Drive.

**Gates (automatic checks):** every file must be a real data file (not an error
page), load correctly, and contain the participant ID `SEQN`. Person-level files
must have exactly one row per person. If anything fails, the notebook stops.

In [1]:
# Starting the project. First, I import the tools I need.
import os, requests
import pandas as pd
from google.colab import drive

# Now I connect my Google Drive, so the raw files are saved permanently.
drive.mount('/content/drive')
RAW = '/content/drive/MyDrive/coffee_lifespan/data/raw'
os.makedirs(RAW, exist_ok=True)

# Next, I list the 8 NHANES cycles I'm using. Each cycle has a letter suffix in its file names.
cycles = {'2003':'C', '2005':'D', '2007':'E', '2009':'F',
          '2011':'G', '2013':'H', '2015':'I', '2017':'J'}

# These are the survey files I need from every cycle.
components = ['DEMO', 'DR1IFF', 'DR2IFF', 'DR1TOT', 'SMQ', 'BMX']

# Here are the CDC web addresses where the files live.
XPT_URL  = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{yr}/DataFiles/{name}.xpt'
MORT_URL = ('https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/'
            'linked_mortality/NHANES_{y1}_{y2}_MORT_2019_PUBLIC.dat')

# I write a small helper that downloads a file, and skips it if I already have it.
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return 'cached'
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    with open(path, 'wb') as f:
        f.write(r.content)
    return 'downloaded'

report = []

# Now I start downloading the survey files, cycle by cycle, and checking each one.
for yr, sfx in cycles.items():
    for comp in components:
        name = f'{comp}_{sfx}'
        path = os.path.join(RAW, f'{name}.xpt')
        status = download(XPT_URL.format(yr=yr, name=name), path)

        # GATE 1: I make sure it's a real data file, not a CDC error page.
        with open(path, 'rb') as f:
            assert f.read(80).startswith(b'HEADER RECORD'), \
                f'GATE FAIL: {name} is not a valid XPT file (check URL)'

        # I open the file to check what's inside.
        df = pd.read_sas(path, format='xport')

        # GATE 2: the file must have rows and the participant ID column.
        assert len(df) > 0, f'GATE FAIL: {name} is empty'
        assert 'SEQN' in df.columns, f'GATE FAIL: {name} has no SEQN'

        # GATE 3: person-level files must have one row per person.
        # (Food files have many rows per person — one per food — so they're skipped here.)
        if comp not in ('DR1IFF', 'DR2IFF'):
            assert df['SEQN'].is_unique, f'GATE FAIL: duplicate SEQN in {name}'

        # I record what I found, then free the memory before the next file.
        report.append({'cycle': f'{yr}-{int(yr)+1}', 'file': name, 'status': status,
                       'rows': len(df), 'people': df['SEQN'].nunique()})
        del df

# Next, I download the death-record files, one per cycle.
for yr in cycles:
    y1, y2 = yr, str(int(yr) + 1)
    name = f'NHANES_{y1}_{y2}_MORT_2019_PUBLIC'
    path = os.path.join(RAW, f'{name}.dat')
    status = download(MORT_URL.format(y1=y1, y2=y2), path)

    with open(path, 'r') as f:
        lines = f.read().splitlines()

    # GATE 4: the death file must have content and must not be an error page.
    assert len(lines) > 0, f'GATE FAIL: {name} is empty'
    assert not lines[0].lstrip().startswith('<'), f'GATE FAIL: {name} looks like HTML'

    report.append({'cycle': f'{y1}-{y2}', 'file': 'MORT_2019', 'status': status,
                   'rows': len(lines), 'people': None})

# Finally, I print a summary of everything I downloaded.
summary = pd.DataFrame(report)
print(summary.to_string(index=False))

# GATE 5: I expect 8 cycles × 6 survey files + 8 death files = 56 files.
assert len(summary) == 56, f'GATE FAIL: expected 56 files, got {len(summary)}'
print(f'\n✅ ALL GATES PASSED — {len(summary)} raw files downloaded and verified.')

Mounted at /content/drive
    cycle      file     status   rows  people
2003-2004    DEMO_C downloaded  10122 10122.0
2003-2004  DR1IFF_C downloaded 131164  9033.0
2003-2004  DR2IFF_C downloaded 120871  8352.0
2003-2004  DR1TOT_C downloaded   9643  9643.0
2003-2004     SMQ_C downloaded   5041  5041.0
2003-2004     BMX_C downloaded   9643  9643.0
2005-2006    DEMO_D downloaded  10348 10348.0
2005-2006  DR1IFF_D downloaded 146940  9349.0
2005-2006  DR2IFF_D downloaded 132151  8429.0
2005-2006  DR1TOT_D downloaded   9950  9950.0
2005-2006     SMQ_D downloaded   7186  7186.0
2005-2006     BMX_D downloaded   9950  9950.0
2007-2008    DEMO_E downloaded  10149 10149.0
2007-2008  DR1IFF_E downloaded 145703  9254.0
2007-2008  DR2IFF_E downloaded 121341  7837.0
2007-2008  DR1TOT_E downloaded   9762  9762.0
2007-2008     SMQ_E downloaded   7145  7145.0
2007-2008     BMX_E downloaded   9762  9762.0
2009-2010    DEMO_F downloaded  10537 10537.0
2009-2010  DR1IFF_F downloaded 150991  9754.0
2009-201

## Step 2 — Read the death records (Linked Mortality Files)

The CDC death files are plain text in **fixed-width** format: each variable sits
in specific character positions on every line (layout from the CDC 2019
Public-Use LMF data dictionary).

| Variable | Positions | Meaning |
|----------|-----------|---------|
| SEQN | 1–6 | Participant ID (links to all NHANES files) |
| ELIGSTAT | 15 | 1 = eligible adult, 2 = under 18 (not released), 3 = ineligible |
| MORTSTAT | 16 | 0 = alive, 1 = died (by 31 Dec 2019) |
| UCOD_LEADING | 17–19 | Underlying cause of death (category code) |
| PERMTH_INT | 43–45 | Months of follow-up from the home interview |
| PERMTH_EXM | 46–48 | Months of follow-up from the exam visit |

**Gates:**
1. Every line has the same length (the file was read correctly)
2. The participant IDs in each death file match the DEMO file **exactly**
3. Codes only take their allowed values
4. Eligible adults always have a death status and follow-up time
5. Follow-up time is never negative and never longer than possible
   (from the start of the cycle to 31 Dec 2019)

In [7]:
# Step 2 begins. I'm now going to read the death-record files.
import numpy as np

PROC = '/content/drive/MyDrive/coffee_lifespan/data/processed'
os.makedirs(PROC, exist_ok=True)

# First, I define where each variable sits on a line, using the CDC data dictionary.
# Python counts from 0, so position 1–6 becomes (0, 6).
colspecs = [(0, 6), (14, 15), (15, 16), (16, 19), (42, 45), (45, 48)]
names    = ['SEQN', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING', 'PERMTH_INT', 'PERMTH_EXM']

mort_all = []

# Now I go through each cycle's death file one by one.
for yr, sfx in cycles.items():
    y1, y2 = yr, str(int(yr) + 1)
    path = os.path.join(RAW, f'NHANES_{y1}_{y2}_MORT_2019_PUBLIC.dat')

    # GATE 1: I check the line structure. The CDC trims empty spaces at the end of lines,
    # so lengths can vary — but no line may be longer than the 48-character layout,
    # and every line must start with a 6-digit participant ID.
    with open(path) as f:
        lines = [line.rstrip('\r\n') for line in f]
    lengths = {len(l) for l in lines}
    assert max(lengths) <= 48, f'GATE FAIL: {yr} has lines longer than 48 characters'
    assert all(l[:6].strip().isdigit() for l in lines), f'GATE FAIL: {yr} has a line without a valid SEQN'

    # For the first cycle only, I print one example line of each length so I can see why they differ.
    if yr == '2003':
        print('Example lines by length (dots mark the end of each line):')
        for n in sorted(lengths):
            example = next(l for l in lines if len(l) == n)
            print(f'  length {n}: {example!r}')
        print()

    # I read the file using the positions. The CDC uses '.' for missing values.
    m = pd.read_fwf(path, colspecs=colspecs, names=names, na_values=['.'])

    # GATE 2: the IDs here must match the DEMO file exactly — same people, no extras, none missing.
    demo_ids = set(pd.read_sas(os.path.join(RAW, f'DEMO_{sfx}.xpt'), format='xport')['SEQN'].astype(int))
    mort_ids = set(m['SEQN'].astype(int))
    assert mort_ids == demo_ids, f'GATE FAIL: {yr} death-file IDs do not match DEMO'
    assert m['SEQN'].is_unique, f'GATE FAIL: duplicate SEQN in {yr} death file'

    # I tag which cycle each person came from, so I can trace them later.
    m['CYCLE'] = f'{y1}-{y2}'
    mort_all.append(m)

# Now I stack all 8 cycles into one table.
mort = pd.concat(mort_all, ignore_index=True)
mort['SEQN'] = mort['SEQN'].astype(int)

# GATE 3: codes must only take their allowed values.
assert set(mort['ELIGSTAT'].dropna().unique()) <= {1, 2, 3}, 'GATE FAIL: unexpected ELIGSTAT code'
assert set(mort['MORTSTAT'].dropna().unique()) <= {0, 1},    'GATE FAIL: unexpected MORTSTAT code'

# GATE 4: every eligible adult must have a death status and a follow-up time.
elig = mort[mort['ELIGSTAT'] == 1]
assert elig['MORTSTAT'].notna().all(),   'GATE FAIL: eligible adults missing MORTSTAT'
assert elig['PERMTH_INT'].notna().all(), 'GATE FAIL: eligible adults missing PERMTH_INT'

# GATE 5: follow-up can't be negative, or much longer than from the cycle start to Dec 2019.
# I found ~1.3% of people go 1–5 months over the window. This comes from inclusive month
# counting and the CDC's privacy noise in the public file, so I allow 6 months of tolerance.
TOLERANCE = 6
start_year = mort['CYCLE'].str[:4].astype(int)
max_possible = (2019 - start_year + 1) * 12          # e.g. 2003 → 204 months
over_by = elig['PERMTH_INT'] - max_possible[elig.index]

assert (elig['PERMTH_INT'] >= 0).all(), 'GATE FAIL: negative follow-up'
assert (over_by <= TOLERANCE).all(), f'GATE FAIL: follow-up exceeds window by more than {TOLERANCE} months'
print(f'Gate 5 note: {(over_by > 0).sum()} people ({(over_by > 0).mean()*100:.2f}%) are 1–{int(over_by.max())} '
      f'months over the window — within tolerance, kept as published by CDC.\n')

# Finally, I summarise what I found, cycle by cycle.
check = (elig.groupby('CYCLE')
             .agg(eligible_adults=('SEQN', 'size'),
                  deaths=('MORTSTAT', 'sum'),
                  max_followup_months=('PERMTH_INT', 'max'))
             .astype(int))
check['death_rate_%'] = (check['deaths'] / check['eligible_adults'] * 100).round(1)
print(check.to_string())
print(f"\nELIGSTAT counts (all {len(mort)} people):\n{mort['ELIGSTAT'].value_counts().sort_index().to_string()}")

# I save the clean death table so the next steps can use it.
mort.to_parquet(os.path.join(PROC, 'mortality_2003_2018.parquet'), index=False)
print('\n✅ ALL GATES PASSED — death records read, verified and saved.')

Example lines by length (dots mark the end of each line):
  length 46: '21006         2.   ..                     .  .'
  length 47: '21015         1100100                     25 24'
  length 48: '21005         10   ..                     201201'

Gate 5 note: 620 people (1.30%) are 1–5 months over the window — within tolerance, kept as published by CDC.

           eligible_adults  deaths  max_followup_months  death_rate_%
CYCLE                                                                
2003-2004             5610    1420                  205          25.3
2005-2006             5561    1027                  180          18.5
2007-2008             6219    1126                  160          18.1
2009-2010             6510     861                  135          13.2
2011-2012             5849     628                  113          10.7
2013-2014             6100     467                   85           7.7
2015-2016             5974     276                   61           4.6
2017-2018   

## Step 3 — Identify coffee in the food records

In NHANES, every food and drink a person reports is stored as an 8-digit USDA
food code. Codes starting with **921** are coffee beverages. The food files only
contain the codes, so we also download each cycle's **food description file**
(`DRXFCD`) to see what each code means.

Here we list every coffee code that appears in our data, with its description and
how often it was reported. This lets us decide what counts as a "cup of coffee"
(for example, dry instant powder can't be converted to cups).

**Gates:**
1. Every 921 code reported in the food files has a description (nothing unknown)
2. Every 921 description is a coffee drink (contains "coffee" or a coffee drink name such as cappuccino, espresso, latte, mocha)
3. Food code and gram columns exist in every food file

In [9]:
# Step 3 begins. Now I'm going to find every coffee drink people reported.

# First, I download the food description files — they translate food codes into words.
for yr, sfx in cycles.items():
    name = f'DRXFCD_{sfx}'
    path = os.path.join(RAW, f'{name}.xpt')
    download(XPT_URL.format(yr=yr, name=name), path)
    with open(path, 'rb') as f:
        assert f.read(80).startswith(b'HEADER RECORD'), f'GATE FAIL: {name} is not a valid XPT file'

coffee_rows = []
descriptions = []

# Now I go through every cycle's food files, day 1 and day 2.
for yr, sfx in cycles.items():
    for day in (1, 2):
        iff = pd.read_sas(os.path.join(RAW, f'DR{day}IFF_{sfx}.xpt'), format='xport')

        # GATE 3: the food code and grams columns must exist.
        code_col, gram_col = f'DR{day}IFDCD', f'DR{day}IGRMS'
        assert code_col in iff.columns and gram_col in iff.columns, \
            f'GATE FAIL: DR{day}IFF_{sfx} is missing {code_col} or {gram_col}'

        # I keep only the foods whose code starts with 921 — coffee.
        codes = iff[code_col].astype('Int64')
        coffee = iff[codes.astype(str).str.startswith('921')]
        coffee_rows.append(pd.DataFrame({'CYCLE': f'{yr}-{int(yr)+1}', 'DAY': day,
                                         'SEQN': coffee['SEQN'].astype(int),
                                         'FOODCODE': coffee[code_col].astype(int),
                                         'GRAMS': coffee[gram_col]}))
        del iff

    # I also read this cycle's food descriptions.
    fcd = pd.read_sas(os.path.join(RAW, f'DRXFCD_{sfx}.xpt'), format='xport', encoding='latin-1')
    descriptions.append(fcd[['DRXFDCD', 'DRXFCLD']].rename(columns={'DRXFDCD': 'FOODCODE',
                                                                    'DRXFCLD': 'DESCRIPTION'}))

# Now I stack all coffee records and all descriptions together.
coffee_all = pd.concat(coffee_rows, ignore_index=True)
desc = (pd.concat(descriptions).astype({'FOODCODE': int})
          .drop_duplicates('FOODCODE', keep='last'))   # keep the newest wording for each code

# Next, I attach a description to every coffee code.
codes_used = (coffee_all.groupby('FOODCODE')
                        .agg(records=('SEQN', 'size'), people=('SEQN', 'nunique'),
                             median_grams=('GRAMS', 'median'))
                        .reset_index()
                        .merge(desc, on='FOODCODE', how='left'))

# GATE 1: every coffee code must have a description.
missing = codes_used[codes_used['DESCRIPTION'].isna()]
assert missing.empty, f'GATE FAIL: codes with no description: {missing["FOODCODE"].tolist()}'

# GATE 2: every description must be a coffee drink. Some coffee drinks don't use the word
# "coffee" (e.g. "Cappuccino, sweetened"), so I check against a list of coffee drink names.
COFFEE_WORDS = 'COFFEE|CAPPUCCINO|ESPRESSO|LATTE|MOCHA|MACCHIATO|AMERICANO|FRAPP|CAFE'
not_coffee = codes_used[~codes_used['DESCRIPTION'].str.upper().str.contains(COFFEE_WORDS)]
assert not_coffee.empty, f'GATE FAIL: 921 codes that are not coffee:\n{not_coffee.to_string()}'

# Finally, I show every coffee code, most common first, and save the list.
pd.set_option('display.max_colwidth', 70)
print(f'Coffee records: {len(coffee_all):,}  |  distinct coffee codes: {len(codes_used)}\n')
print(codes_used.sort_values('records', ascending=False)
                [['FOODCODE', 'records', 'median_grams', 'DESCRIPTION']]
                .to_string(index=False))

coffee_all.to_parquet(os.path.join(PROC, 'coffee_records_raw.parquet'), index=False)
codes_used.to_csv(os.path.join(PROC, 'coffee_codes_list.csv'), index=False)
print('\n✅ ALL GATES PASSED — coffee records identified and saved.')

Coffee records: 54,458  |  distinct coffee codes: 101

 FOODCODE  records  median_grams                                                                                                      DESCRIPTION
 92101000    33339       345.000                                                                                                   Coffee, brewed
 92103000     6945       253.300                                                                                   Coffee, instant, reconstituted
 92111010     4254       251.600                                                                                    Coffee, brewed, decaffeinated
 92114000     2045       240.000                                                                    Coffee, instant, decaffeinated, reconstituted
 92101500     1116       355.200                                                               Coffee, brewed, blend of regular and decaffeinated
 92100500     1092       326.000                                     

## Step 4 — Daily coffee intake per person

**Rules:**
- **Excluded:** 10 dry powder / concentrate codes (weighed before water was added,
  median 0.9–12.8 g, so they can't be converted to cups). ~300 of 54,458 records.
- **Included:** all other 921 codes — brewed, instant (made up), decaf, espresso,
  lattes, iced and frozen drinks. Decaf is flagged separately.
- **1 cup = 8 fl oz = 236.6 g** of coffee drink.
- **Two recall days:** NHANES marks each day as reliable or not. If both days are
  reliable we average them; if only day 1 is reliable we use day 1. People without
  a reliable day 1 are left out.
- People with a reliable recall but no coffee = **0 cups** (non-drinkers).

**Limitation:** cups are measured by volume, so an espresso (~90 g) counts as
~0.4 cups and the milk in a latte counts toward volume. These drinks are <3% of records.

**Gates:**
1. Exactly 10 codes are excluded, and all have median weight < 20 g
2. Every included code has median weight > 40 g (a real drink)
3. Grams per person add up **exactly** to the raw records they came from
4. Spot-check: for 5 random coffee drinkers, recompute intake directly from the
   raw food file and confirm it matches
5. One row per person; no negative values

In [10]:
# Step 4 begins. Now I turn individual coffee records into each person's daily intake.
import random
CUP_GRAMS = 236.6

# First, I attach the description to every coffee record.
recs = coffee_all.merge(desc, on='FOODCODE', how='left')
D = recs['DESCRIPTION'].str.upper()

# Now I mark the dry powders and concentrates that can't be converted into cups.
recs['EXCLUDE'] = (D.str.contains('NOT RECONSTITUTED') | D.str.contains('DRY')
                   | (D == 'COFFEE, LIQUID CONCENTRATE'))
recs['DECAF'] = D.str.contains('DECAF')

# GATE 1: exactly 10 codes excluded, and all of them must be tiny (< 20 g).
excl = codes_used[codes_used['FOODCODE'].isin(recs.loc[recs['EXCLUDE'], 'FOODCODE'])]
assert len(excl) == 10, f'GATE FAIL: expected 10 excluded codes, got {len(excl)}'
assert (excl['median_grams'] < 20).all(), 'GATE FAIL: an excluded code is a full-size drink'

# GATE 2: every code I keep must be a real drink (> 40 g).
kept = codes_used[~codes_used['FOODCODE'].isin(excl['FOODCODE'])]
assert (kept['median_grams'] > 40).all(), f'GATE FAIL: tiny code kept:\n{kept[kept["median_grams"] <= 40]}'
print(f'Excluded {len(excl)} codes ({recs["EXCLUDE"].sum()} records):')
print(excl[['FOODCODE', 'records', 'median_grams', 'DESCRIPTION']].to_string(index=False), '\n')

# Next, I add up each person's coffee grams per day (total, and decaf separately).
recs = recs[~recs['EXCLUDE']].copy()
recs['DECAF_G'] = recs['GRAMS'].where(recs['DECAF'], 0)
per_day = (recs.groupby(['SEQN', 'DAY'])[['GRAMS', 'DECAF_G']].sum()
               .unstack('DAY', fill_value=0))
per_day.columns = [f'{v}_D{d}' for v, d in per_day.columns]   # GRAMS_D1, GRAMS_D2, DECAF_G_D1, ...
per_day = per_day.reset_index()

# Now I find out who had a reliable recall each day (status code 1 = reliable).
status = []
for yr, sfx in cycles.items():
    name = f'DR2TOT_{sfx}'
    download(XPT_URL.format(yr=yr, name=name), os.path.join(RAW, f'{name}.xpt'))
    t1 = pd.read_sas(os.path.join(RAW, f'DR1TOT_{sfx}.xpt'), format='xport')[['SEQN', 'DR1DRSTZ']]
    t2 = pd.read_sas(os.path.join(RAW, f'DR2TOT_{sfx}.xpt'), format='xport')[['SEQN', 'DR2DRSTZ']]
    s = t1.merge(t2, on='SEQN', how='left')
    s['CYCLE'] = f'{yr}-{int(yr)+1}'
    status.append(s)
status = pd.concat(status, ignore_index=True)
status['SEQN'] = status['SEQN'].astype(int)
status['D1_OK'] = status['DR1DRSTZ'] == 1
status['D2_OK'] = status['DR2DRSTZ'] == 1

# I keep only people with a reliable day 1, and attach their coffee (no record = 0 g).
person = status[status['D1_OK']].merge(per_day, on='SEQN', how='left').fillna(
    {'GRAMS_D1': 0, 'GRAMS_D2': 0, 'DECAF_G_D1': 0, 'DECAF_G_D2': 0})

# Now I average the two days when both are reliable, otherwise I use day 1.
both = person['D2_OK']
person['COFFEE_G']  = np.where(both, (person['GRAMS_D1'] + person['GRAMS_D2']) / 2, person['GRAMS_D1'])
person['DECAF_G']   = np.where(both, (person['DECAF_G_D1'] + person['DECAF_G_D2']) / 2, person['DECAF_G_D1'])
person['COFFEE_CUPS'] = person['COFFEE_G'] / CUP_GRAMS
person['N_DAYS'] = np.where(both, 2, 1)

# GATE 3: grams per person must add up exactly to the raw records they came from.
raw_d1 = recs.loc[(recs['DAY'] == 1) & recs['SEQN'].isin(person['SEQN']), 'GRAMS'].sum()
d2_people = person.loc[both, 'SEQN']
raw_d2 = recs.loc[(recs['DAY'] == 2) & recs['SEQN'].isin(d2_people), 'GRAMS'].sum()
assert np.isclose(person['GRAMS_D1'].sum(), raw_d1), 'GATE FAIL: day 1 grams do not reconcile'
assert np.isclose(person.loc[both, 'GRAMS_D2'].sum(), raw_d2), 'GATE FAIL: day 2 grams do not reconcile'

# GATE 4: I spot-check 5 random coffee drinkers by going back to the raw food file itself.
random.seed(42)
excluded_codes = set(excl['FOODCODE'])
suffix_of = {f'{y}-{int(y)+1}': s for y, s in cycles.items()}
for seqn in random.sample(person.loc[person['COFFEE_G'] > 0, 'SEQN'].tolist(), 5):
    row = person[person['SEQN'] == seqn].iloc[0]
    sfx = suffix_of[row['CYCLE']]
    def raw_grams(day):
        iff = pd.read_sas(os.path.join(RAW, f'DR{day}IFF_{sfx}.xpt'), format='xport')
        iff = iff[iff['SEQN'] == seqn]
        codes = iff[f'DR{day}IFDCD'].astype(int)
        is_coffee = codes.astype(str).str.startswith('921') & ~codes.isin(excluded_codes)
        return iff.loc[is_coffee, f'DR{day}IGRMS'].sum()
    expected = (raw_grams(1) + raw_grams(2)) / 2 if row['D2_OK'] else raw_grams(1)
    assert np.isclose(row['COFFEE_G'], expected), f'GATE FAIL: SEQN {seqn} → {row["COFFEE_G"]} vs raw {expected}'
    print(f'Spot-check SEQN {seqn}: {row["COFFEE_G"]:.1f} g = raw {expected:.1f} g ✓')

# GATE 5: one row per person, nothing negative.
assert person['SEQN'].is_unique, 'GATE FAIL: duplicate people'
assert (person[['COFFEE_G', 'DECAF_G']] >= 0).all().all(), 'GATE FAIL: negative intake'

# Finally, I summarise coffee drinking and save the table.
drinkers = person['COFFEE_G'] > 0
print(f'\nPeople with a reliable day-1 recall: {len(person):,}')
print(f'  with 2 reliable days: {both.sum():,}  |  day 1 only: {(~both).sum():,}')
print(f'Coffee drinkers: {drinkers.sum():,} ({drinkers.mean()*100:.1f}%)')
print(f'Cups/day among drinkers — median {person.loc[drinkers, "COFFEE_CUPS"].median():.2f}, '
      f'90th percentile {person.loc[drinkers, "COFFEE_CUPS"].quantile(0.9):.2f}')

keep_cols = ['SEQN', 'CYCLE', 'N_DAYS', 'COFFEE_G', 'COFFEE_CUPS', 'DECAF_G']
person[keep_cols].to_parquet(os.path.join(PROC, 'coffee_per_person.parquet'), index=False)
print('\n✅ ALL GATES PASSED — daily coffee intake calculated and saved.')

Excluded 10 codes (300 records):
 FOODCODE  records  median_grams                                                                                    DESCRIPTION
 92105000        1           5.0                                                                     Coffee, liquid concentrate
 92191000        6           0.9                                  Coffee, dry instant powder, NS as to regular or decaffeinated
 92191100      197           0.9                                                             Coffee, instant, not reconstituted
 92191200       66           0.9                                              Coffee, instant, decaffeinated, not reconstituted
 92191250        1           2.7                                                                  Coffee, dry, acid neutralized
 92191400        3          10.2                                   Coffee, instant, pre-sweetened with sugar, not reconstituted
 92192000        5          12.8          Coffee, mocha, instant, pre-l

## Step 5 — Build the study cohort

We combine coffee intake, demographics and death records, then apply the
inclusion rules one at a time, counting how many people each rule removes
(this becomes the study flowchart).

**Inclusion rules:**
1. Has a reliable day-1 diet recall (from Step 4)
2. Aged 20 or older at the exam
3. Eligible for death-record linkage (ELIGSTAT = 1)
4. Not pregnant at the exam (pregnancy changes coffee habits)
5. Has exam-based follow-up time (diet was recorded at the exam, so the
   follow-up clock starts there: `PERMTH_EXM`)

**Gates:**
1. Every person after each rule is a subset of the step before (no one appears from nowhere)
2. The flowchart adds up: start − all exclusions = final cohort
3. Final cohort: one row per person, all aged ≥ 20, all ELIGSTAT = 1
4. Deaths in the cohort match a fresh count taken directly from the saved death file

In [11]:
# Step 5 begins. Now I bring coffee, demographics and death records together.

# First, I load what I saved in earlier steps.
coffee = pd.read_parquet(os.path.join(PROC, 'coffee_per_person.parquet'))
mort   = pd.read_parquet(os.path.join(PROC, 'mortality_2003_2018.parquet'))

# Next, I collect age, sex and pregnancy status from every DEMO file.
demo = []
for yr, sfx in cycles.items():
    d = pd.read_sas(os.path.join(RAW, f'DEMO_{sfx}.xpt'), format='xport')
    demo.append(d[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDEXPRG']])
demo = pd.concat(demo, ignore_index=True)
demo['SEQN'] = demo['SEQN'].astype(int)
assert demo['SEQN'].is_unique, 'GATE FAIL: duplicate SEQN across DEMO files'

# Now I join everything into one table, starting from people with a reliable diet recall.
df = (coffee.merge(demo, on='SEQN', how='left')
            .merge(mort.drop(columns='CYCLE'), on='SEQN', how='left'))
assert len(df) == len(coffee), 'GATE FAIL: merging changed the number of people'

# Now I apply the inclusion rules one by one and record the flowchart.
flow = [('Reliable day-1 diet recall', len(df))]
prev_ids = set(df['SEQN'])

def apply_rule(data, mask, label):
    global prev_ids
    kept = data[mask].copy()
    # GATE 1: the people I keep must all come from the previous step.
    assert set(kept['SEQN']) <= prev_ids, f'GATE FAIL: new people appeared at "{label}"'
    prev_ids = set(kept['SEQN'])
    flow.append((label, len(kept)))
    return kept

df = apply_rule(df, df['RIDAGEYR'] >= 20,       'Aged 20+')
df = apply_rule(df, df['ELIGSTAT'] == 1,        'Eligible for death linkage')
df = apply_rule(df, df['RIDEXPRG'] != 1,        'Not pregnant')
df = apply_rule(df, df['PERMTH_EXM'].notna(),   'Has exam-based follow-up')

# I print the flowchart, showing how many people each rule removed.
print('Study flowchart')
for i, (label, n) in enumerate(flow):
    removed = '' if i == 0 else f'   (removed {flow[i-1][1] - n:,})'
    print(f'  {label:<32} {n:>7,}{removed}')

# GATE 2: the flowchart must add up.
total_removed = sum(flow[i-1][1] - flow[i][1] for i in range(1, len(flow)))
assert flow[0][1] - total_removed == len(df), 'GATE FAIL: flowchart does not add up'

# GATE 3: one row per person, everyone 20+, everyone eligible.
assert df['SEQN'].is_unique,          'GATE FAIL: duplicate people in cohort'
assert (df['RIDAGEYR'] >= 20).all(),  'GATE FAIL: under-20 in cohort'
assert (df['ELIGSTAT'] == 1).all(),   'GATE FAIL: ineligible person in cohort'

# GATE 4: deaths in my cohort must match a fresh count from the saved death file.
fresh = mort[mort['SEQN'].isin(df['SEQN'])]
assert int(df['MORTSTAT'].sum()) == int(fresh['MORTSTAT'].sum()), 'GATE FAIL: death count mismatch'

# Finally, I describe the cohort.
df['FOLLOWUP_YRS'] = df['PERMTH_EXM'] / 12
drinkers = df['COFFEE_CUPS'] > 0
print(f'\nFinal cohort: {len(df):,} adults')
print(f'Deaths: {int(df["MORTSTAT"].sum()):,} ({df["MORTSTAT"].mean()*100:.1f}%)')
print(f'Median follow-up: {df["FOLLOWUP_YRS"].median():.1f} years')
print(f'Coffee drinkers: {drinkers.sum():,} ({drinkers.mean()*100:.1f}%)')
print(f'Age range: {int(df["RIDAGEYR"].min())}–{int(df["RIDAGEYR"].max())}')

df.to_parquet(os.path.join(PROC, 'cohort_base.parquet'), index=False)
print('\n✅ ALL GATES PASSED — study cohort built and saved.')

Study flowchart
  Reliable day-1 diet recall        69,535
  Aged 20+                          39,757   (removed 29,778)
  Eligible for death linkage        39,682   (removed 75)
  Not pregnant                      38,814   (removed 868)
  Has exam-based follow-up          38,814   (removed 0)

Final cohort: 38,814 adults
Deaths: 4,994 (12.9%)
Median follow-up: 8.0 years
Coffee drinkers: 22,861 (58.9%)
Age range: 20–85

✅ ALL GATES PASSED — study cohort built and saved.


## Step 6 — Add confounders (factors that could mislead the result)

Coffee drinkers differ from non-drinkers in age, smoking, income and more. To
avoid crediting coffee for something else, we add these factors and adjust for
them in the analysis.

| Variable | Source | Coding |
|----------|--------|--------|
| Age, sex | DEMO | years; male/female |
| Race/ethnicity | DEMO `RIDRETH1` | Mexican American, Other Hispanic, NH White, NH Black, Other |
| Education | DEMO `DMDEDUC2` | < high school, high school, some college, college+ |
| Income-to-poverty ratio | DEMO `INDFMPIR` | continuous (0–5) |
| Smoking | SMQ `SMQ020`, `SMQ040` | never / former / current |
| BMI | BMX `BMXBMI` | measured kg/m² |
| Calories, alcohol | DR1TOT / DR2TOT | daily average, same days as coffee |
| Survey design | DEMO, DR1TOT | strata, PSU, dietary weight ÷ 8 cycles |

**Limitation:** physical activity is not included, because NHANES changed its
activity questions in 2007 and the measure isn't comparable across all cycles.

**Gates:**
1. Merging never changes the number of people
2. Every recoded category reconciles exactly with the raw survey codes
3. Smoking logic is consistent (no "current smoker" who never smoked)
4. Calories/alcohol averages are recalculated and match
5. Values are in plausible ranges (BMI 10–150, income ratio 0–5). One participant
   has a measured BMI of 130 — extreme but real, so kept.
6. No "fake zeros" remain. Pandas can read a true SAS zero as a tiny number
   (e.g. 5e-79); without fixing this, ~27,500 non-drinkers of alcohol would
   not be counted as zero.

In [17]:
# Step 6 begins. Now I add the factors that could confuse the coffee–death link.

# First, I collect the extra variables from every cycle.
extra = []
for yr, sfx in cycles.items():
    d  = pd.read_sas(os.path.join(RAW, f'DEMO_{sfx}.xpt'),   format='xport')[
            ['SEQN', 'RIDRETH1', 'DMDEDUC2', 'INDFMPIR', 'SDMVPSU', 'SDMVSTRA']]
    t1 = pd.read_sas(os.path.join(RAW, f'DR1TOT_{sfx}.xpt'), format='xport')[
            ['SEQN', 'DR1TKCAL', 'DR1TALCO', 'WTDRD1']]
    t2 = pd.read_sas(os.path.join(RAW, f'DR2TOT_{sfx}.xpt'), format='xport')[
            ['SEQN', 'DR2TKCAL', 'DR2TALCO']]
    sm = pd.read_sas(os.path.join(RAW, f'SMQ_{sfx}.xpt'),    format='xport')[['SEQN', 'SMQ020', 'SMQ040']]
    bm = pd.read_sas(os.path.join(RAW, f'BMX_{sfx}.xpt'),    format='xport')[['SEQN', 'BMXBMI']]
    extra.append(d.merge(t1, on='SEQN', how='left').merge(t2, on='SEQN', how='left')
                  .merge(sm, on='SEQN', how='left').merge(bm, on='SEQN', how='left'))
extra = pd.concat(extra, ignore_index=True)
extra['SEQN'] = extra['SEQN'].astype(int)

# Now I fix a known pandas quirk: when reading SAS files, a true zero can come back
# as a tiny number like 5e-79. I convert these back to exact zeros.
num = extra.select_dtypes('number').columns
fake_zeros = ((extra[num].abs() < 1e-10) & (extra[num] != 0)).sum().sum()
extra[num] = extra[num].mask(extra[num].abs() < 1e-10, 0)
print(f'Fixed {fake_zeros:,} fake-zero values (SAS reading quirk).\n')

# Now I attach them to my cohort.
cohort = pd.read_parquet(os.path.join(PROC, 'cohort_base.parquet'))
n_before = len(cohort)
cohort = cohort.merge(extra, on='SEQN', how='left')

# GATE 1: merging must not change the number of people.
assert len(cohort) == n_before, 'GATE FAIL: merge changed cohort size'

# Next, I recode the categories into readable labels.
cohort['SEX'] = cohort['RIAGENDR'].map({1: 'Male', 2: 'Female'})
cohort['RACE'] = cohort['RIDRETH1'].map({1: 'Mexican American', 2: 'Other Hispanic',
                                         3: 'NH White', 4: 'NH Black', 5: 'Other'})
cohort['EDUCATION'] = cohort['DMDEDUC2'].map({1: '< High school', 2: '< High school',
                                              3: 'High school', 4: 'Some college', 5: 'College+'})
# (codes 7 = refused and 9 = don't know become missing)

# Now I build smoking status from two questions:
# SMQ020 = smoked 100+ cigarettes in life (1 yes, 2 no); SMQ040 = smoke now (1 daily, 2 some days, 3 not at all).
cohort['SMOKING'] = np.select(
    [cohort['SMQ020'] == 2,
     (cohort['SMQ020'] == 1) & cohort['SMQ040'].isin([1, 2]),
     (cohort['SMQ020'] == 1) & (cohort['SMQ040'] == 3)],
    ['Never', 'Current', 'Former'], default=None)

# GATE 2: every recoded category must match the raw codes exactly.
def reconcile(new_col, raw_col, mapping):
    for label, codes in mapping.items():
        n_new = (cohort[new_col] == label).sum()
        n_raw = cohort[raw_col].isin(codes).sum()
        assert n_new == n_raw, f'GATE FAIL: {new_col}="{label}" has {n_new}, raw codes give {n_raw}'
reconcile('SEX', 'RIAGENDR', {'Male': [1], 'Female': [2]})
reconcile('RACE', 'RIDRETH1', {'Mexican American': [1], 'Other Hispanic': [2], 'NH White': [3],
                               'NH Black': [4], 'Other': [5]})
reconcile('EDUCATION', 'DMDEDUC2', {'< High school': [1, 2], 'High school': [3],
                                    'Some college': [4], 'College+': [5]})
assert (cohort['SMOKING'] == 'Never').sum() == (cohort['SMQ020'] == 2).sum(), 'GATE FAIL: never-smoker count'

# GATE 3: smoking logic must be consistent.
assert not ((cohort['SMOKING'] == 'Current') & (cohort['SMQ020'] == 2)).any(), 'GATE FAIL: current smoker who never smoked'

# Now I average calories and alcohol over the same days I used for coffee.
two = cohort['N_DAYS'] == 2
for var, d1, d2 in [('KCAL', 'DR1TKCAL', 'DR2TKCAL'), ('ALCOHOL_G', 'DR1TALCO', 'DR2TALCO')]:
    cohort[var] = np.where(two, (cohort[d1] + cohort[d2]) / 2, cohort[d1])
    # GATE 4: I recompute for one-day and two-day people separately and compare.
    assert np.allclose(cohort.loc[two, var], (cohort.loc[two, d1] + cohort.loc[two, d2]) / 2, equal_nan=True), f'GATE FAIL: {var} 2-day'
    assert np.allclose(cohort.loc[~two, var], cohort.loc[~two, d1], equal_nan=True), f'GATE FAIL: {var} 1-day'

# Next, I create the pooled survey weight: 8 cycles combined, so I divide by 8.
cohort['WEIGHT'] = cohort['WTDRD1'] / 8

# GATE 5: values must be in plausible ranges.
# (BMI max is 130.21 — one real, measured person with severe obesity, so I allow up to 150.)
assert cohort['BMXBMI'].dropna().between(10, 150).all(),   'GATE FAIL: implausible BMI'
assert cohort['INDFMPIR'].dropna().between(0, 5).all(),    'GATE FAIL: income ratio out of 0–5'
assert (cohort['KCAL'].dropna() >= 0).all(),               'GATE FAIL: negative calories'

# GATE 6: no fake zeros may remain in any numeric column.
num_c = cohort.select_dtypes('number').columns
assert not ((cohort[num_c].abs() < 1e-10) & (cohort[num_c] != 0)).any().any(), 'GATE FAIL: fake zeros remain'

# Finally, I check how much data is missing for each factor.
covars = ['RIDAGEYR', 'SEX', 'RACE', 'EDUCATION', 'INDFMPIR', 'SMOKING', 'BMXBMI', 'KCAL', 'ALCOHOL_G', 'WEIGHT']
miss = cohort[covars].isna().sum().to_frame('missing')
miss['missing_%'] = (miss['missing'] / len(cohort) * 100).round(1)
print('Missing data per factor:')
print(miss.to_string())

complete = cohort[covars].notna().all(axis=1)
print(f'\nPeople with all factors present: {complete.sum():,} of {len(cohort):,} ({complete.mean()*100:.1f}%)')
print(f'Deaths among them: {int(cohort.loc[complete, "MORTSTAT"].sum()):,}')
print(f'\nCalories — median {cohort["KCAL"].median():.0f}, '
      f'1st pct {cohort["KCAL"].quantile(0.01):.0f}, 99th pct {cohort["KCAL"].quantile(0.99):.0f}')

cohort.to_parquet(os.path.join(PROC, 'cohort_covariates.parquet'), index=False)
print('\n✅ ALL GATES PASSED — confounders added and saved.')

Fixed 118,116 fake-zero values (SAS reading quirk).

Missing data per factor:
           missing  missing_%
RIDAGEYR         0        0.0
SEX              0        0.0
RACE             0        0.0
EDUCATION       40        0.1
INDFMPIR      3130        8.1
SMOKING         24        0.1
BMXBMI         460        1.2
KCAL             0        0.0
ALCOHOL_G        0        0.0
WEIGHT           0        0.0

People with all factors present: 35,224 of 38,814 (90.8%)
Deaths among them: 4,447

Calories — median 1900, 1st pct 612, 99th pct 4729

✅ ALL GATES PASSED — confounders added and saved.


## Step 7 — Final analysis cohort and coffee groups

Two final rules:
6. **Complete data on all confounders** (90.8% of people; income is the most
   often missing, at 8.1%)
7. **Plausible calorie intake** — standard Willett rule: women 500–3,500 kcal/day,
   men 800–4,200 kcal/day. Outside this range, a diet report is unlikely to reflect
   real eating.

Then everyone is placed into a coffee group (1 cup = 8 oz):

| Group | Cups per day |
|-------|-------------|
| None | 0 |
| ≤1 cup | > 0 to 1 |
| 1–2 cups | > 1 to 2 |
| 2–3 cups | > 2 to 3 |
| > 3 cups | more than 3 |

**Gates:**
1. Flowchart adds up, and each rule only removes people (never adds)
2. No missing values remain in any variable used in the analysis
3. Every person's coffee group matches their exact cups/day (boundaries verified)
4. Group sizes add up to the full cohort
5. Death count matches a fresh count from the raw death file

In [18]:
# Step 7 begins. Now I apply the last two rules and create the coffee groups.
cohort = pd.read_parquet(os.path.join(PROC, 'cohort_covariates.parquet'))
covars = ['RIDAGEYR', 'SEX', 'RACE', 'EDUCATION', 'INDFMPIR', 'SMOKING', 'BMXBMI', 'KCAL', 'ALCOHOL_G', 'WEIGHT']

# First, I continue the flowchart from Step 5.
flow2 = [('Cohort from Step 5', len(cohort))]
prev = set(cohort['SEQN'])

def rule(data, mask, label):
    global prev
    kept = data[mask].copy()
    # GATE 1: each rule may only remove people.
    assert set(kept['SEQN']) <= prev, f'GATE FAIL: new people appeared at "{label}"'
    prev = set(kept['SEQN'])
    flow2.append((label, len(kept)))
    return kept

# Now I keep only people with every confounder present.
cohort = rule(cohort, cohort[covars].notna().all(axis=1), 'Complete data on confounders')

# Next, I remove implausible diet reports using the Willett calorie rule.
ok_kcal = (((cohort['SEX'] == 'Female') & cohort['KCAL'].between(500, 3500)) |
           ((cohort['SEX'] == 'Male')   & cohort['KCAL'].between(800, 4200)))
cohort = rule(cohort, ok_kcal, 'Plausible calories (Willett)')

# I print the flowchart.
print('Study flowchart (continued)')
for i, (label, n) in enumerate(flow2):
    removed = '' if i == 0 else f'   (removed {flow2[i-1][1] - n:,})'
    print(f'  {label:<32} {n:>7,}{removed}')
assert flow2[0][1] - sum(flow2[i-1][1] - flow2[i][1] for i in range(1, len(flow2))) == len(cohort), \
    'GATE FAIL: flowchart does not add up'

# GATE 2: nothing used in the analysis may be missing.
analysis_vars = covars + ['COFFEE_CUPS', 'MORTSTAT', 'PERMTH_EXM', 'SDMVPSU', 'SDMVSTRA']
assert cohort[analysis_vars].notna().all().all(), 'GATE FAIL: missing values remain'

# Now I sort everyone into coffee groups.
labels = ['None', '≤1 cup', '1–2 cups', '2–3 cups', '>3 cups']
cups = cohort['COFFEE_CUPS']
cohort['COFFEE_GROUP'] = np.select(
    [cups == 0, cups <= 1, cups <= 2, cups <= 3, cups > 3], labels, default=None)
cohort['COFFEE_GROUP'] = pd.Categorical(cohort['COFFEE_GROUP'], categories=labels, ordered=True)

# GATE 3: every group's cups must sit inside its boundaries.
bounds = {'None': (0, 0), '≤1 cup': (0, 1), '1–2 cups': (1, 2), '2–3 cups': (2, 3), '>3 cups': (3, np.inf)}
for g, (lo, hi) in bounds.items():
    c = cohort.loc[cohort['COFFEE_GROUP'] == g, 'COFFEE_CUPS']
    if g == 'None':
        assert (c == 0).all(), 'GATE FAIL: non-zero cups in "None"'
    else:
        assert (c > lo).all() and (c <= hi).all(), f'GATE FAIL: "{g}" has cups outside ({lo}, {hi}]'

# GATE 4: the groups must add up to the whole cohort.
assert cohort['COFFEE_GROUP'].notna().all(), 'GATE FAIL: someone has no coffee group'
assert cohort['COFFEE_GROUP'].value_counts().sum() == len(cohort), 'GATE FAIL: groups do not add up'

# GATE 5: deaths must match a fresh count from the raw death file.
mort = pd.read_parquet(os.path.join(PROC, 'mortality_2003_2018.parquet'))
assert int(cohort['MORTSTAT'].sum()) == int(mort[mort['SEQN'].isin(cohort['SEQN'])]['MORTSTAT'].sum()), \
    'GATE FAIL: death count mismatch'

# Finally, I take a first (unadjusted) look at each coffee group.
summary = (cohort.groupby('COFFEE_GROUP', observed=True)
                 .agg(people=('SEQN', 'size'),
                      deaths=('MORTSTAT', 'sum'),
                      median_age=('RIDAGEYR', 'median'),
                      current_smokers_pct=('SMOKING', lambda s: (s == 'Current').mean() * 100),
                      median_cups=('COFFEE_CUPS', 'median')))
summary['deaths'] = summary['deaths'].astype(int)
summary['death_%'] = (summary['deaths'] / summary['people'] * 100).round(1)
summary['current_smokers_pct'] = summary['current_smokers_pct'].round(1)
summary['median_cups'] = summary['median_cups'].round(2)
print(f'\nFinal cohort: {len(cohort):,} adults | Deaths: {int(cohort["MORTSTAT"].sum()):,} | '
      f'Median follow-up: {(cohort["PERMTH_EXM"]/12).median():.1f} years\n')
print(summary.to_string())

# I save the analysis-ready file for Notebook 2.
final_cols = ['SEQN', 'CYCLE', 'COFFEE_CUPS', 'COFFEE_GROUP', 'COFFEE_G', 'DECAF_G',
              'MORTSTAT', 'UCOD_LEADING', 'PERMTH_EXM', 'RIDAGEYR', 'SEX', 'RACE', 'EDUCATION',
              'INDFMPIR', 'SMOKING', 'BMXBMI', 'KCAL', 'ALCOHOL_G', 'WEIGHT', 'SDMVPSU', 'SDMVSTRA']
cohort[final_cols].to_parquet(os.path.join(PROC, 'analysis_cohort.parquet'), index=False)
print('\n✅ ALL GATES PASSED — analysis cohort saved. Notebook 1 (cleaning) is complete.')

Study flowchart (continued)
  Cohort from Step 5                38,814
  Complete data on confounders      35,224   (removed 3,590)
  Plausible calories (Willett)      33,976   (removed 1,248)

Final cohort: 33,976 adults | Deaths: 4,295 | Median follow-up: 8.1 years

              people  deaths  median_age  current_smokers_pct  median_cups  death_%
COFFEE_GROUP                                                                       
None           13814    1230        41.0                 18.9         0.00      8.9
≤1 cup          5362     697        51.0                 16.4         0.63     13.0
1–2 cups        7313    1143        56.0                 18.2         1.39     15.6
2–3 cups        3856     631        56.0                 21.8         2.35     16.4
>3 cups         3631     594        55.0                 36.5         3.97     16.4

✅ ALL GATES PASSED — analysis cohort saved. Notebook 1 (cleaning) is complete.
